<a href="https://colab.research.google.com/github/ciceromayk/arkrender/blob/main/colab/arkitekt_comfyui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARKITEKT — ComfyUI grátis no Colab

Sobe um servidor ComfyUI com Flux.1-dev + ControlNet Union (depth) numa GPU
grátis do Colab e expõe uma URL pública temporária. Use essa URL como
`ARKITEKT_COMFY_URL` no `.env` local ou nos secrets do app Streamlit.

**Os modelos (~25 GB) ficam salvos no seu Google Drive.** Na primeira vez
baixa tudo (leva alguns minutos); da segunda vez em diante o notebook
detecta que já está no Drive e **pula direto para o servidor** — sem
rebaixar nada, sem pedir login do Hugging Face de novo.

**Antes de rodar (só na primeira vez):**
- Ative a GPU: menu **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.
- Crie um token em https://huggingface.co/settings/tokens e aceite a licença em
  https://huggingface.co/black-forest-labs/FLUX.1-dev (o modelo é *gated*, sem isso o download falha).

**Limites do plano grátis:** sessão cai depois de ~12h (ou bem antes, por
ociosidade), e a URL do túnel muda a cada vez que você reinicia este
notebook — não é uma URL fixa de produção. Os modelos no Drive sobrevivem
normalmente à queda da sessão.

## 1. Checar GPU

In [1]:
!nvidia-smi

Sat Aug 29 20:11:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Instalar ComfyUI

In [2]:
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 1219, done.
remote: Counting objects: 100% (1219/1219), done.
remote: Compressing objects: 100% (1051/1051), done.
remote: Total 1219 (delta 128), reused 1034 (delta 125), pack-reused 0 (from 0)
Receiving objects: 100% (1219/1219), 11.74 MiB | 15.06 MiB/s, done.
Resolving deltas: 100% (128/128), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.3/342.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 MB 12.3

## 3. Nós extras — preprocessador de depth (Depth Anything V2)

Necessário para gerar o mapa de profundidade a partir do screenshot dentro
do próprio ComfyUI (a rota fal.ai faz isso num endpoint separado; aqui
precisamos do nó equivalente).

In [3]:
%cd /content/ComfyUI/custom_nodes
!git clone --depth 1 https://github.com/Fannovel16/comfyui_controlnet_aux
%cd comfyui_controlnet_aux
!pip install -q -r requirements.txt
%cd /content/ComfyUI

/content/ComfyUI/custom_nodes
Cloning into 'comfyui_controlnet_aux'...
remote: Enumerating objects: 876, done.
remote: Counting objects: 100% (876/876), done.
remote: Compressing objects: 100% (758/758), done.
remote: Total 876 (delta 117), reused 699 (delta 89), pack-reused 0 (from 0)
Receiving objects: 100% (876/876), 37.76 MiB | 16.09 MiB/s, done.
Resolving deltas: 100% (117/117), done.
/content/ComfyUI/custom_nodes/comfyui_controlnet_aux
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:

## 4. Google Drive — pasta permanente dos modelos

Roda **sempre**, mesmo nas próximas vezes: monta o Drive e faz `models/`
apontar pra lá. Se os arquivos já existirem de uma sessão anterior, as
próximas células não baixam nada de novo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
cache = pathlib.Path('/content/drive/MyDrive/arkitekt_comfy_models')
local = pathlib.Path('/content/ComfyUI/models')
for sub in ['unet', 'vae', 'clip', 'controlnet']:
    (cache / sub).mkdir(parents=True, exist_ok=True)
    if (local / sub).exists() and not (local / sub).is_symlink():
        for f in (local / sub).iterdir():
            f.rename(cache / sub / f.name)
        (local / sub).rmdir()
    if not (local / sub).exists():
        (local / sub).symlink_to(cache / sub)
print("models/ aponta para o Drive — o que for baixado aqui fica salvo para sempre")

## 5. Conferir o que falta

Se a saída disser "nada para baixar", pule as células 6 e 7 e vá direto
para a célula 8 (subir o servidor).

In [ ]:
import pathlib

ARQUIVOS = {
    "models/unet/flux1-dev-fp8.safetensors":
        "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
    "models/vae/ae.safetensors":
        "https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/ae.safetensors",
    "models/clip/clip_l.safetensors":
        "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors",
    "models/clip/t5xxl_fp8_e4m3fn.safetensors":
        "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors",
    "models/controlnet/flux-controlnet-union.safetensors":
        "https://huggingface.co/InstantX/FLUX.1-dev-Controlnet-Union/resolve/main/diffusion_pytorch_model.safetensors",
}

faltando = {
    p: u for p, u in ARQUIVOS.items()
    if not pathlib.Path(p).exists() or pathlib.Path(p).stat().st_size == 0
}

if faltando:
    print(f"faltam {len(faltando)} de {len(ARQUIVOS)} arquivo(s) — rode as células 6 e 7")
    for p in faltando:
        print(" -", p)
else:
    print("tudo já está no Drive — pule para a célula 8 (subir o servidor)")

## 6. Login no Hugging Face — só roda se faltar algo

FLUX.1-dev é *gated*: gere um token em
https://huggingface.co/settings/tokens e antes disso aceite a licença em
https://huggingface.co/black-forest-labs/FLUX.1-dev.

In [ ]:
if faltando:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("nada para baixar — login não é necessário")

## 7. Baixar só o que falta — só roda se faltar algo

In [ ]:
import subprocess

for destino, url in faltando.items():
    print(f"baixando {destino} ...")
    subprocess.run(["wget", "-q", "--show-progress", "-O", destino, url], check=True)

print("modelos prontos no Drive" if faltando else "nada para baixar")

## 8. Subir o servidor ComfyUI em background

In [ ]:
import subprocess, time
proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
    cwd='/content/ComfyUI', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
time.sleep(15)
print("ComfyUI subindo — confira a célula 9 para a URL pública")

## 9. Expor com túnel público (cloudflared — sem cadastro)

Copie a URL `https://xxxx.trycloudflare.com` impressa abaixo. Ela serve
tanto pra abrir a interface do ComfyUI no navegador (montar o workflow,
próximo passo) quanto pra `ARKITEKT_COMFY_URL` do ARKITEKT.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

import subprocess
tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in tunnel.stdout:
    print(line, end='')
    if 'trycloudflare.com' in line:
        print('\n>>> copie a URL acima (https://....trycloudflare.com) <<<')
        break

## 10. Próximo passo

**Primeira vez:**
1. Abra a URL do túnel no seu navegador — é a interface do ComfyUI.
2. Monte o workflow uma única vez seguindo `docs/comfyui_gratis.md` do repo
   ARKITEKT (lista exata de nós e como conectá-los) e exporte em
   **Save (API Format)** como `flux_depth.json`.
3. Coloque o arquivo em `workflows/flux_depth.json` no repo e confira os
   ids em `core/engines/comfy_engine.py` (dict `NODE`).

**Toda vez (inclusive a primeira):**

4. Exporte `ARKITEKT_COMFY_URL` (ou cole no app Streamlit / secrets) com a
   URL do túnel e rode `python bench/run.py --engines comfy` ou o app.

Mantenha esta aba do Colab aberta — fechar encerra o servidor e derruba a
URL. Da próxima vez, reabra este notebook e rode as células 1 → 4 → 8 → 9
(pulando 5/6/7 se a célula 5 disser que não falta nada) — o workflow
montado no passo 2 não precisa ser refeito, ele mora no repo, não no Colab.